# 🎨 BrandMind AI — Week 5
## Colour Palette & Visual Harmony Engine
**Tools:** OpenCV · scikit-learn (KMeans) · Matplotlib

In [ ]:
!pip install opencv-python-headless scikit-learn matplotlib numpy pandas pillow -q

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from PIL import Image
import io, os
print('Ready')

## 1. KMeans Colour Extraction

In [ ]:
def extract_palette(image_input, n_colors=5):
    """Extract dominant colours from logo image using KMeans."""
    if isinstance(image_input, str):
        img = cv2.imread(image_input)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = image_input  # numpy array

    pixels = img.reshape(-1, 3).astype(float)
    kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=10)
    kmeans.fit(pixels)
    centers = kmeans.cluster_centers_.astype(int)
    proportions = np.bincount(kmeans.labels_) / len(kmeans.labels_)
    # Sort by proportion (most dominant first)
    order = np.argsort(proportions)[::-1]
    hex_codes = ['#%02x%02x%02x' % tuple(centers[i]) for i in order]
    rgb_codes = [tuple(centers[i]) for i in order]
    props = proportions[order]
    return hex_codes, rgb_codes, props

# Demo with synthetic image
np.random.seed(42)
demo_img = np.random.randint(0, 255, (100, 100, 3), dtype=np.uint8)
hex_codes, rgb_codes, proportions = extract_palette(demo_img, n_colors=5)
print('Extracted palette:')
for h, r, p in zip(hex_codes, rgb_codes, proportions):
    print(f'  {h}  RGB{r}  ({p*100:.1f}%)')

## 2. Colour-Personality & Industry Mapping

In [ ]:
colour_personality = {
    'blue':   {'industries': ['Technology','Finance','Healthcare'], 'traits': ['Trust','Professional','Calm']},
    'green':  {'industries': ['Sustainability','Healthcare','Food & Beverage'], 'traits': ['Growth','Natural','Fresh']},
    'red':    {'industries': ['Food & Beverage','Retail','Energy'], 'traits': ['Energy','Passion','Urgency']},
    'gold':   {'industries': ['Finance','Luxury','Fashion'], 'traits': ['Premium','Success','Quality']},
    'black':  {'industries': ['Fashion','Technology','Luxury'], 'traits': ['Sophistication','Power','Elegance']},
    'orange': {'industries': ['Retail','Travel','Food & Beverage'], 'traits': ['Friendly','Creative','Enthusiastic']},
    'purple': {'industries': ['Education','Healthcare','Beauty'], 'traits': ['Wisdom','Creativity','Royalty']},
}

# Predefined palettes per industry
industry_palettes = {
    'Technology':    ['#1e2a4a','#7b9ef0','#eef2ff','#94a3b8'],
    'Finance':       ['#1a1a18','#b8975a','#f5f0e8','#6b6a65'],
    'Healthcare':    ['#0f4c81','#4fc3f7','#e3f2fd','#78909c'],
    'Fashion':       ['#2a1835','#c084fc','#f8f0ff','#9c27b0'],
    'Sustainability':['#2d4a3e','#a8d5c2','#f0f7f4','#8fbc8f'],
    'Retail':        ['#3d1a00','#f97316','#fff8f0','#ff8c42'],
    'Food & Beverage':['#4e342e','#ff8f00','#fff8e1','#a1887f'],
    'Travel':        ['#01579b','#00b0ff','#e1f5fe','#4dd0e1'],
}

palette_df = pd.DataFrame([
    {'Industry': ind, 'Primary': cols[0], 'Accent': cols[1], 'Background': cols[2], 'Neutral': cols[3]}
    for ind, cols in industry_palettes.items()
])
print(palette_df.to_string(index=False))
palette_df.to_csv('industry_colour_mapping.csv', index=False)
print('\nSaved: industry_colour_mapping.csv')

## 3. Visualisation — Colour Swatches

In [ ]:
def plot_palette(hex_colors, title='Brand Palette', proportions=None):
    fig, ax = plt.subplots(figsize=(10, 2.5))
    n = len(hex_colors)
    for i, hex_c in enumerate(hex_colors):
        width = proportions[i] if proportions is not None else 1/n
        rect = mpatches.FancyBboxPatch([sum(proportions[:i]) if proportions is not None else i/n, 0.15],
                                        width, 0.7,
                                        boxstyle='round,pad=0.01',
                                        facecolor=hex_c, edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        ax.text((sum(proportions[:i]) if proportions is not None else i/n) + width/2,
                0.05, hex_c, ha='center', va='center', fontsize=8, color='#333')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    return fig

# Display all industry palettes
fig, axes = plt.subplots(len(industry_palettes), 1, figsize=(12, len(industry_palettes)*1.5))
for ax, (ind, cols) in zip(axes, industry_palettes.items()):
    for i, c in enumerate(cols):
        rect = mpatches.FancyBboxPatch([i*0.25, 0.1], 0.22, 0.8,
                                        boxstyle='round,pad=0.02',
                                        facecolor=c, edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        ax.text(i*0.25+0.11, 0.05, c, ha='center', fontsize=7, color='#555')
    ax.text(-0.02, 0.5, ind, ha='right', va='center', fontsize=9, fontweight='bold', transform=ax.transAxes)
    ax.set_xlim(-0.01, 1); ax.set_ylim(0, 1); ax.axis('off')
plt.suptitle('Industry Colour Palettes', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('industry_colour_palettes.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved: industry_colour_palettes.png')

In [ ]:
# Save extracted palettes as downloadable PNG swatches
os.makedirs('outputs', exist_ok=True)
for ind, cols in list(industry_palettes.items())[:3]:
    fig = plot_palette(cols, title=f'{ind} Brand Palette')
    fig.savefig(f'outputs/palette_{ind.replace(" ","_").lower()}.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
print('Colour swatches saved to outputs/')

## Week 5 Complete ✅
- ✅ KMeans colour extraction (HEX + RGB + proportions)
- ✅ Industry-to-colour mapping table (CSV)
- ✅ Colour swatch visualisation (PNG)
- ✅ Palette preview ready for Week 6 animation